# Homework 4

**Course:** MATH 6388

**Author:** Brady Lamson

**Date:** 9/29/2026

---

## Problem 3.1

Consider the sigmoid function

$$
p=\sigma(a)=\frac{1}{1+e^{-a}},
\qquad a\in\mathbb{R}.
$$

### Part A

**Problem:**

Show that the log-odds satisfy

$$
\log\left(\frac{p}{1-p}\right)=a.
$$

**Solution:**

Before we start, we will simplify the denominator as that will save us a lot of time later.

$$
\begin{align}
    1 - p &= 1 - \frac{1}{1 + e^{-a}} \\
    &= \frac{1 + e^{-a}}{1 + e^{-a}} - \frac{1}{1 + e^{-a}} & \text{(Rewrite 1)} \\
    &= \frac{1 + e^{-a} - 1}{1 + e^{-a}} & \text{(Combine fractions)} \\
    &= \frac{e^{-a}}{1 + e^{-a}} \\
    &= \frac{1}{e^a (1+e^{-a})} \\
    1 - p &= \frac{1}{1+e^a} = (1+e^a)^{-1}
\end{align}
$$

Okay, now we start properly.

$$
\begin{align}
    \ln\left( \frac{p}{1-p} \right) &= \ln\left( \frac{\sigma(a)}{1-p} \right) \\
    &= \ln\left( \frac{(1 + e^{-a})^{-1} }{(1 + e^a)^{-1}}\right) & \text{(From earlier result)} \\
    &= \ln\left( \frac{1 + e^a}{1 + e^{-a}}\right) & \text{(Flip fraction)}
\end{align}
$$

Now we simplify $1 + e^{-a}$

$$
\begin{align}
    1 + e^{-a} &= \frac{e^a}{e^a} + \frac{1}{e^a} & \text{(Rewrite 1)} \\
    &= \frac{e^a + 1}{e^a}
\end{align}
$$

Now we return to where we were before

$$
\begin{align}
    \ln\left( \frac{p}{1-p} \right) &= \ln\left( \frac{1 + e^a}{1 + e^{-a}}\right) \\
    &= \ln\left( \frac{1 + e^a}{\frac{e^a + 1}{e^a}}\right) & \text{(From earlier result)} \\
    &= \ln\left( \frac{e^a (1 + e^a)}{1+e^a} \right) & \text{(Move } e^a \text{ to numerator)} \\
    &= \ln(e^a) & \text{(Simplify)} \\
    &= a & \text{(Log properties)}
\end{align}
$$

Completing the problem for part a.

### Part B

**Problem:**

Show that the derivative of the sigmoid function can be written as

$$
\sigma'(a)=\sigma(a)\bigl[1-\sigma(a)\bigr].
$$

**Solution:**

$$
\begin{align}
    \frac{d}{da} \sigma(a) &= \frac{d}{da} \frac{1}{1+e^{-a}} \\
    &= -(1+e^{-a})^{-2} \cdot \frac{d}{da} 1+e^{-a} \\
    &= -(1+e^{-a})^{-2} \cdot -e^{-a} \\
    &= \frac{1}{e^a(1+e^{-a})^2} \\
    &= \frac{1}{1+e^{-a}} \cdot \frac{1}{1+e^{-a}} \cdot \frac{1}{e^a} & \text{(Split up fraction)} \\
    &= \sigma(a) \cdot \frac{1}{1+e^{-a}} \cdot \frac{1}{e^a} & \text{(Substitution)} \\
    &= \sigma(a) \cdot \frac{1}{e^a(1+e^{-a})} \\
    &= \sigma(a) \cdot \frac{1}{1+e^a} \\
    &= \sigma(a) \cdot (1 - \sigma(a)) & \text{(From part A)}
\end{align}
$$

The final line is justified by the beginning of the work done in **part a** during the simplification of $1 - p$. It saved us time in both parts!

This completes the work for part b.

## Problem 3.2

Consider the binary classifier trained on the synthetic dataset above.
In this problem, keep the trained model fixed and use its test-set
probabilities to explore how error costs affect the predicted labels.

Recall that the cost-based decision rule predicts class 1 when

$$
p \geq \frac{C_{FP}}{C_{FP}+C_{FN}}.
$$

---

### Set up the model

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=.7,
    random_state=1
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Standardize using training-set statistics.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

# Match the model's output shape: (number of observations, 1).
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [2]:
torch.manual_seed(42)

model = nn.Linear(
    in_features=2,
    out_features=1
)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

In [3]:
n_epochs = 1000

model.train()

for epoch in range(n_epochs):
    # Forward pass: raw logits.
    logits = model(X_train)

    # Binary cross-entropy directly from logits.
    loss = loss_fn(logits, y_train)

    # Compute gradients and update the parameters.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1:3d} | Loss: {loss.item():.4f}")

Epoch 100 | Loss: 0.4583
Epoch 200 | Loss: 0.4264
Epoch 300 | Loss: 0.4173
Epoch 400 | Loss: 0.4138
Epoch 500 | Loss: 0.4122
Epoch 600 | Loss: 0.4115
Epoch 700 | Loss: 0.4112
Epoch 800 | Loss: 0.4110
Epoch 900 | Loss: 0.4109
Epoch 1000 | Loss: 0.4109


In [4]:
model.eval()

with torch.no_grad():
    test_logits = model(X_test)
    test_probabilities = torch.sigmoid(test_logits)
    test_predictions = (test_probabilities >= 0.5).float()

    accuracy = (test_predictions == y_test).float().mean()

print(f"Test accuracy: {accuracy.item():.2%}")

Test accuracy: 80.00%


---

### Part A

**Problem**

Suppose a false negative is four times as costly as a false
positive: $C_{FN}=4C_{FP}$, where $C_{FP}>0$.

Determine the decision threshold and use it to generate predicted labels
from `test_probabilities`. Compare these predictions with `y_test` and
report:

- The number of false positives: predicted label 1, true label 0.
- The number of false negatives: predicted label 0, true label 1.

**Solution:**

$$
\begin{align}
    p &\geq \frac{C_{FP}}{C_{FP}+C_{FN}} \\
    &\geq \frac{C_{FP}}{5 C_{FP}} \\
    &\geq \frac{1}{5} \\
\end{align}
$$


Our new threshold then is 0.20.

In [17]:
with torch.no_grad():
    threshold = 0.20
    test_predictions = (test_probabilities >= threshold).float()
    false_positives = ((test_predictions == 1) & (y_test == 0)).sum().int()
    false_negatives = ((test_predictions == 0) & (y_test == 1)).sum().int()
    print(f"Threshold: {threshold}\nFalse positives: {false_positives}\nFalse negatives: {false_negatives}")

Threshold: 0.2
False positives: 8
False negatives: 2


### Part B

**Problem:**

Now suppose a false positive is four times as costly as a false
negative: $C_{FP}=4C_{FN}$, where $C_{FN}>0$.

Determine the new threshold, generate predicted labels, and report the
numbers of false positives and false negatives.

**Solution:**

$$
\begin{align}
    p &\geq \frac{C_{FP}}{C_{FP}+C_{FN}} \\
    &\geq \frac{4 C_{FN}}{5 C_{FN}} \\
    &\geq \frac{4}{5}
\end{align}
$$

In [18]:
with torch.no_grad():
    threshold = 4/5
    test_predictions = (test_probabilities >= threshold).float()
    false_positives = ((test_predictions == 1) & (y_test == 0)).sum().int()
    false_negatives = ((test_predictions == 0) & (y_test == 1)).sum().int()
    print(f"Threshold: {threshold}\nFalse positives: {false_positives}\nFalse negatives: {false_negatives}")

Threshold: 0.8
False positives: 1
False negatives: 12


### Part C

**Problem:**

Summarize your results for both cost settings and the original
threshold of 0.5 in a table with columns for the decision threshold,
number of false positives, and number of false negatives.

Use the same test probabilities in every case. Do not retrain the model
or choose a threshold by searching for the best test-set performance.
Include your code and results.

**Solution:**

So just make a markdown table of the results from the previous parts? There should be no code here aside from getting the FP and FN count for the original threshold.

|Decision Threshold| N False Positive| N False Negative|
|---|---|---|
|0.5|3|5|
|0.2|8|2|
|0.8|1|12|

In [16]:
# Collecting 0.5 results
with torch.no_grad():
    threshold = 0.5
    test_predictions = (test_probabilities >= threshold).float()
    false_positives = ((test_predictions == 1) & (y_test == 0)).sum().int()
    false_negatives = ((test_predictions == 0) & (y_test == 1)).sum().int()
    print(f"Threshold: {threshold}\nFalse positives: {false_positives}\nFalse negatives: {false_negatives}")

Threshold: 0.5
False positives: 3
False negatives: 5
